# 🎯 RAG-Based Resume Profile Matching System
### Complete Experimentation, Retrieval Analysis, and System Benchmarks

This notebook provides a step-by-step walkthrough and technical analysis of the **RAG-Based Profile Matching System**.

### 🌟 System Architecture Overview
1. **Multi-Format Extraction**: Parses `.pdf`, `.docx`, and `.txt` candidate resumes (`fs_tools.py`).
2. **Section-Aware Chunker**: Parses header sections (`Experience`, `Skills`, `Summary`, `Education`), calculates experience years, extracts canonical skill casing, and slices into 512-token sliding window chunks (`chunker.py`).
3. **Dense Vector Storage**: Pinecone Serverless vector database (`resume-profile-matcher`) at 768 dimensions using Cosine Similarity (`init_pinecone.py`).
4. **Gemini Matryoshka Embeddings**: Uses `gemini-embedding-001` with `output_dimensionality=768`, persistent local SHA-256 caching (`.embedding_cache.json`), and multi-item batching (`vector_store.py`).
5. **Hybrid Retrieval**: Combines Cosine Similarity ($S_{semantic}$) with Sparse Keyword Match Density ($S_{keyword}$) using RRF scoring (`retriever.py`).
6. **Hard Constraint Filtering Engine**: Enforces strict minimum experience years check and mandatory skill presence check (`matcher.py`).
7. **0–100 Quantitative Relevance Scoring**: Computes a transparent, 4-part weighted match score (`matcher.py`).
8. **Single-Batch LLM Match Reasoner**: Generates candidate evaluation rationale via Google Gemini in a single batch prompt (`llm_reasoning.py`).

## 1. Environment Setup & Package Imports

In [1]:
import os
import sys
import json
import time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

# Ensure project root and src directory are in sys.path
project_root = Path.cwd()
src_dir = project_root / "src"
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from chunker import SectionAwareChunker
from vector_store import GeminiEmbedder, PineconeVectorStore
from retriever import HybridRetriever
from matcher import ProfileMatcher
from llm_reasoning import LLMReasoningEngine

load_dotenv()
print("✅ Environment & Dependencies Loaded Successfully!")


✅ Environment & Dependencies Loaded Successfully!


## 2. Dataset Exploration (50 Resumes & 10 Job Descriptions)

In [2]:
resumes_dir = Path("resumes")
jds_dir = Path("job_descriptions")

resume_files = list(resumes_dir.glob("*"))
jd_files = list(jds_dir.glob("*.txt"))

print(f"📁 Total Resume Dataset Files: {len(resume_files)}")
print(f"📄 Total Job Description Files: {len(jd_files)}\n")

format_counts = {".pdf": 0, ".docx": 0, ".txt": 0}
for f in resume_files:
    ext = f.suffix.lower()
    if ext in format_counts:
        format_counts[ext] += 1

print("📊 Resume Format Distribution:")
for fmt, cnt in format_counts.items():
    print(f"   - {fmt.upper()}: {cnt} resumes")

📁 Total Resume Dataset Files: 50
📄 Total Job Description Files: 10

📊 Resume Format Distribution:
   - .PDF: 17 resumes
   - .DOCX: 17 resumes
   - .TXT: 16 resumes


## 3. Section-Aware Chunking Analysis

In [3]:
chunker = SectionAwareChunker()
chunks = chunker.chunk_directory("resumes")

unique_candidates = {c.candidate_id for c in chunks}
print(f"✅ Parsed {len(chunks)} section chunks across {len(unique_candidates)} unique candidate profiles.\n")

sample_chunk = chunks[0]
print(f"Sample Candidate: {sample_chunk.candidate_name} ({sample_chunk.candidate_id})")
print(f"Section Type:     {sample_chunk.section_type}")
print(f"Experience:       {sample_chunk.experience_years} years")
print(f"Extracted Skills: {sample_chunk.skills_list}")
print(f"Text Snippet:\n{sample_chunk.text[:200]}...")

✅ Parsed 245 section chunks across 50 unique candidate profiles.

Sample Candidate: Alice Johnson (cand_alice_johnson)
Section Type:     Professional Summary
Experience:       5.0 years
Extracted Skills: ['Airflow', 'BERT', 'HuggingFace', 'Julia', 'NumPy', 'Pandas', 'PyTorch', 'Python', 'R', 'SQL', 'Scikit-Learn', 'Snowflake', 'SpaCy', 'Spark', 'Tableau', 'Transformers', 'XGBoost']
Text Snippet:
[Candidate: Alice Johnson | Section: Professional Summary | Exp: 5.0 yrs]
Alice Johnson - Senior Data Scientist
Contact: alice.johnson@datascience.ne...


## 4. Pinecone Vector Database & Embedding Inspection

In [4]:
embedder = GeminiEmbedder()
store = PineconeVectorStore()

test_text = "Senior AI Engineer with Python, FastAPI, and Pinecone experience."
vector = embedder.embed_text(test_text)

print(f"✅ Vector Dimension: {len(vector)} (Expected: 768)")
print(f"Vector First 5 Values: {vector[:5]}")

[CACHE] Loaded 269 cached vector embeddings from '.embedding_cache.json'.
[INFO] Checking Pinecone index status for: 'resume-profile-matcher'...
[INFO] Index 'resume-profile-matcher' already exists.
[INFO] Pinecone Index Stats:
       - Total Vectors Indexed: 246
       - Vector Dimension: 768
✅ Vector Dimension: 768 (Expected: 768)
Vector First 5 Values: [-0.024072316, 0.0011949611, 0.01828576, -0.07927625, -0.036295254]


## 5. Profile Matching Execution & Hard Constraint Verification

In [5]:
matcher = ProfileMatcher()

sample_jd_path = "job_descriptions/jd_01_senior_ai_systems_engineer.txt"
with open(sample_jd_path, "r", encoding="utf-8") as f:
    sample_jd_text = f.read()

t0 = time.time()
result = matcher.match_candidates(sample_jd_text, top_k_candidates=5)
elapsed = round(time.time() - t0, 2)

jd_summary = result["job_description_summary"]
print(f"🎯 Target Position: {jd_summary['title']}")
print(f"⏱  Execution Time: {elapsed} seconds")
print(f"📊 Total Evaluated: {result['total_candidates_evaluated']} | Passed Filters: {result['candidates_passed_filters']} | Failed: {result['candidates_failed_filters']}\n")

df_matches = pd.DataFrame([
    {
        "Rank": m["rank"],
        "Candidate Name": m["candidate_name"],
        "Match Score": f"{m['match_score']}/100",
        "Passed Filter": "✅ PASSED" if m["passed_mandatory_filters"] else "❌ FAILED",
        "Must-Have Matched": ", ".join(m["matched_skills"]["must_have"]),
        "Missing Skills": ", ".join(m["missing_skills"])
    }
    for m in result["top_matches"]
])

df_matches

[CACHE] Loaded 270 cached vector embeddings from '.embedding_cache.json'.
[INFO] Checking Pinecone index status for: 'resume-profile-matcher'...
[INFO] Index 'resume-profile-matcher' already exists.
[INFO] Pinecone Index Stats:
       - Total Vectors Indexed: 246
       - Vector Dimension: 768
🎯 Target Position: Senior AI & LLM Systems Engineer
⏱  Execution Time: 0.82 seconds
📊 Total Evaluated: 14 | Passed Filters: 1 | Failed: 13



,Rank,Candidate Name,Match Score,Passed Filter,Must-Have Matched,Missing Skills
0,1,John Doe,92/100,✅ PASSED,"Docker, FastAPI, LangChain, Pinecone, PyTorch,...",
1,2,Zoe Saldana,71/100,❌ FAILED,"Docker, PyTorch, Python","FastAPI, LangChain, Pinecone, ChromaDB"
2,3,Rachel Green,70/100,❌ FAILED,"Docker, PyTorch, Python","FastAPI, LangChain, Pinecone, ChromaDB"
3,4,Daniel Craig,63/100,❌ FAILED,"Docker, PyTorch, Python","FastAPI, LangChain, Pinecone, AWS, ChromaDB, K..."
4,5,Olivia Wilde,62/100,❌ FAILED,"Docker, PyTorch, Python","FastAPI, LangChain, Pinecone, AWS, ChromaDB, K..."


## 6. System Performance Metrics: Retrieval Accuracy & Latency Benchmarks

In [6]:
benchmark_results = []
total_start = time.time()

for idx, jd_path in enumerate(sorted(jds_dir.glob("*.txt")), 1):
    with open(jd_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    t_start = time.time()
    res = matcher.match_candidates(text, top_k_candidates=5)
    t_diff = round(time.time() - t_start, 2)
    
    summary = res["job_description_summary"]
    top_cand = res["top_matches"][0] if res["top_matches"] else None
    
    benchmark_results.append({
        "JD #": idx,
        "Job Title": summary["title"],
        "Min Exp": f"{summary['min_experience_years']} yrs",
        "Top Match Candidate": top_cand["candidate_name"] if top_cand else "N/A",
        "Top Match Score": f"{top_cand['match_score']}/100" if top_cand else "0/100",
        "Filter Pass Ratio": f"{res['candidates_passed_filters']}/{res['total_candidates_evaluated']}",
        "Latency (sec)": t_diff
    })

total_time = round(time.time() - total_start, 2)
df_bench = pd.DataFrame(benchmark_results)

print(f"⚡ Benchmark Execution Summary across 10 JDs:")
print(f"   - Total Benchmark Time: {total_time} seconds")
print(f"   - Average Latency per Match: {round(total_time / 10, 2)} seconds\n")

df_bench

⚡ Benchmark Execution Summary across 10 JDs:
   - Total Benchmark Time: 3.2 seconds
   - Average Latency per Match: 0.32 seconds



,JD #,Job Title,Min Exp,Top Match Candidate,Top Match Score,Filter Pass Ratio,Latency (sec)
0,1,Senior AI & LLM Systems Engineer,5.0 yrs,John Doe,92/100,1/14,0.32
1,2,Lead Frontend Web Developer,4.0 yrs,Jane Smith,86/100,2/14,0.32
2,3,Senior Cloud DevOps & Infrastructure Architect,6.0 yrs,Bob Williams,90/100,1/13,0.32
3,4,Senior Data Engineer & Pipeline Architect,5.0 yrs,Evan Wright,92/100,2/15,0.33
4,5,Embedded Systems & Firmware Specialist,3.0 yrs,Rosamund Pike,89/100,6/15,0.33
5,6,Lead Cybersecurity & Incident Response Engineer,5.0 yrs,Charlie Brown,87/100,15/18,0.32
6,7,Lead Cross-Platform Mobile App Engineer,3.0 yrs,Quentin Tarantino,90/100,6/14,0.28
7,8,Junior/Mid-level Python Backend Developer,1.0 yrs,Nathaniel Vance,91/100,2/22,0.33
8,9,Senior NLP Research Scientist,4.0 yrs,Olivia Wilde,90/100,3/13,0.28
9,10,Senior Database Administrator & Storage Engineer,6.0 yrs,Patrick Stewart,93/100,3/12,0.28


## 7. Gemini LLM Match Reasoning Generation

In [7]:
reasoning_engine = LLMReasoningEngine()

top_matches = result["top_matches"]
rationales = reasoning_engine.generate_candidate_reasoning_batch(top_matches, jd_summary)

print("🧠 Generated LLM Candidate Rationales:\n")
for cand, rationale in zip(top_matches[:3], rationales[:3]):
    print(f"👤 {cand['candidate_name']} ({cand['match_score']}/100):")
    print(f"   {rationale}\n" + "-" * 70)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


🧠 Generated LLM Candidate Rationales:

👤 John Doe (92/100):
   John Doe is a highly qualified candidate who exceeds the minimum experience requirements and possesses the full suite of mandatory technical skills. His specialized background in LangChain and Pinecone makes him an ideal fit for the Senior AI & LLM Systems Engineer role.
----------------------------------------------------------------------
👤 Zoe Saldana (71/100):
   Zoe Saldana meets the minimum experience requirement but lacks essential LLM-focused skills including FastAPI, LangChain, and Pinecone. Consequently, she does not currently meet the mandatory technical constraints for this position.
----------------------------------------------------------------------
👤 Rachel Green (70/100):
   Rachel Green possesses relevant MLOps experience but fails to meet the specific technical prerequisites required for this LLM-focused role. Her profile lacks the mandatory proficiency in FastAPI, LangChain, and Pinecone necessary to qu